# Page 9 (dev): Best-of-N with a Process Reward Model

Dev only. `Qwen2.5-Math-PRM-7B` scores every step of every chain in the temperature sweep, and Best-of-N keeps the chain it scores highest. This notebook generates nothing: it reads the sweep's `chains_T*_L2048.json` files.

Built for a Colab A100.

### Mount Google Drive for Saving Results
Run this cell to connect your Google Drive. Everything is saved to a folder named `Page9_PRM_Dev` in your Drive. Outside Colab it goes to a local folder of the same name.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/Page9_PRM_Dev'
    COLAB = True
except ImportError:
    SAVE_DIR = 'Page9_PRM_Dev'
    COLAB = False

os.makedirs(SAVE_DIR, exist_ok=True)
os.chdir(SAVE_DIR)
print(f"Working directory: {os.getcwd()}")

### Step 1: Install Dependencies, Imports and Configurations

Every number the run uses is set here.

In [ ]:
#@title Installation and Setup
!pip install -q datasets transformers accelerate math_verify safetensors

import json
import math
import re
import time
from collections import defaultdict
from functools import cache
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from math_verify import parse, verify
from matplotlib.ticker import PercentFormatter
from safetensors import safe_open
from tqdm.auto import tqdm
from transformers import AutoTokenizer, Qwen2Model

PRM_MODEL = "Qwen/Qwen2.5-Math-PRM-7B"
STEP_SEP = "<extra_0>"   # the token the PRM scores at, one per step
SYSTEM = "Please reason step by step, and put your final answer within \\boxed{}."

TEMPERATURES = [round(0.1 * i, 1) for i in range(12)]   # 0.0 to 1.1
CAP = 2048                   # the sweep's tokens per chain
N_PROBLEMS = 100             # problems every temperature has
N_CHAINS = 16                # chains per problem
BEST_OF_SIZES = [1, 2, 4, 8, 16]
N_DRAWS = 100                # random sets of chains averaged per point
SEED = 42

TOKENS_PER_BATCH = 32_768    # padded tokens per scoring pass; lower it if memory runs out
DEVICE = "cuda"

### Step 2: Load the Chains and the Problems

Put the 12 `chains_T*_L2048.json` files in the Drive folder, or select them when this cell asks. MATH-500 supplies each problem's text and answer, joined on `unique_id`.

T 0.0 and 0.1 hold more problems than the other files. All of their chains are scored, and accuracy uses the 100 problems every temperature has.

In [ ]:
names = {t: f"chains_T{t:.1f}_L{CAP}.json" for t in TEMPERATURES}

missing = [n for n in names.values() if not os.path.exists(n)]
if missing and COLAB:
    from google.colab import files
    print(f"Select the {len(missing)} missing files.")
    files.upload()
    missing = [n for n in names.values() if not os.path.exists(n)]
assert not missing, f"not in {os.getcwd()}: {missing}"

traces = {}
for t, name in names.items():
    with open(name) as f:
        traces[t] = json.load(f)
    assert all(len(r["chains"]) == N_CHAINS for r in traces[t]), f"{name}: not {N_CHAINS} chains each"

dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
question = {row["unique_id"]: row["problem"] for row in dataset}

shared = set.intersection(*[{r["unique_id"] for r in traces[t]} for t in TEMPERATURES])
assert len(shared) == N_PROBLEMS, f"{len(shared)} problems are in every file, not {N_PROBLEMS}"
problems = [row for row in dataset if row["unique_id"] in shared]   # dataset order, as the sweep drew them

total = sum(len(r["chains"]) for t in TEMPERATURES for r in traces[t])
print(f"{total:,} chains across {len(TEMPERATURES)} temperatures")
print(f"{len(problems)} problems shared by every temperature")

### Step 3: Answers and Sets of Chains

The grader and the random sets are the sweep's own, so every point here sits on the same chains as its vote.

- **Two answers are the same** when `math_verify` says they are equal.
- **A chain with no boxed answer** has nothing to submit.
- **The sets of N chains** are every combination when there are at most `N_DRAWS`, otherwise `N_DRAWS` random ones, drawn from one generator in the sweep's order.

In [ ]:
@cache
def grade(pred, gold):
    gt = str(gold).strip()
    if pred == "":
        return False
    try:
        return bool(verify(parse(f"${gt}$"), parse(f"${pred}$")))
    except Exception:
        return pred.replace(" ", "").lower() == gt.replace(" ", "").lower()


@cache
def same_answer(a, b):
    if a.replace(" ", "") == b.replace(" ", ""):
        return True
    try:
        return bool(verify(parse(f"${a}$"), parse(f"${b}$")))
    except Exception:
        return False


def tally(chains, gold):
    # Groups the chains' answers into distinct answers. Returns each chain's group
    # (None when it has no answer) and whether each group is the right answer.
    groups, labels = [], []
    for chain in chains:
        answer = chain["answer"]
        if answer == "":
            labels.append(None)
            continue
        for g, rep in enumerate(groups):
            if same_answer(rep, answer):
                labels.append(g)
                break
        else:
            groups.append(answer)
            labels.append(len(groups) - 1)
    return labels, [grade(rep, gold) for rep in groups]


def chain_sets(n, rng):
    # Which n of the 16 chains compete: every combination when there are at most
    # N_DRAWS of them, otherwise N_DRAWS random ones.
    if math.comb(N_CHAINS, n) <= N_DRAWS:
        return [list(c) for c in combinations(range(N_CHAINS), n)]
    return [sorted(rng.choice(N_CHAINS, n, replace=False).tolist()) for _ in range(N_DRAWS)]

### Step 4: Load the PRM

The checkpoint's bundled modeling code returns finite but wrong scores on current `transformers`, about 0.25 for every step, and raises nothing. So the body loads as transformers' own `Qwen2Model` and the checkpoint's score head is attached by hand: same weights, none of the bundled code.

A step is one paragraph, cut at every blank line. A step's score is the probability the PRM gives it of being sound.

The model card publishes four step scores for one worked example. This cell reproduces them and stops the run if it cannot.

In [ ]:
path = snapshot_download(PRM_MODEL)
tok = AutoTokenizer.from_pretrained(path)
body = Qwen2Model.from_pretrained(path, torch_dtype=torch.bfloat16).to(DEVICE).eval()
assert next(body.parameters()).dtype == torch.bfloat16

with open(f"{path}/model.safetensors.index.json") as f:
    shard = json.load(f)["weight_map"]
head_state = {}
for name in ("0.weight", "0.bias", "2.weight", "2.bias"):
    with safe_open(f"{path}/{shard['score.' + name]}", framework="pt") as f:
        head_state[name] = f.get_tensor("score." + name).float()
width = body.config.hidden_size
head = torch.nn.Sequential(torch.nn.Linear(width, width), torch.nn.ReLU(), torch.nn.Linear(width, 2))
head.load_state_dict(head_state)
head = head.to(DEVICE).float().eval()

SEP_ID = tok.encode(STEP_SEP)[0]
PAD_ID = tok.pad_token_id
assert SEP_ID != PAD_ID


def split_steps(text):
    steps = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    return steps or [text.strip() or "(empty)"]


def sequence(problem, steps):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": STEP_SEP.join(steps) + STEP_SEP},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return tok(text, add_special_tokens=False).input_ids


@torch.inference_mode()
def prm_batch(sequences):
    # One list of step scores per sequence.
    longest = max(len(s) for s in sequences)
    input_ids = torch.full((len(sequences), longest), PAD_ID)
    mask = torch.zeros((len(sequences), longest), dtype=torch.long)
    for row, seq in enumerate(sequences):
        input_ids[row, : len(seq)] = torch.tensor(seq)
        mask[row, : len(seq)] = 1
    input_ids, mask = input_ids.to(DEVICE), mask.to(DEVICE)
    hidden = body(input_ids=input_ids, attention_mask=mask, use_cache=False).last_hidden_state
    at_step = input_ids == SEP_ID
    probs = torch.softmax(head(hidden[at_step].float()), dim=-1)[:, 1].cpu()
    return [p.tolist() for p in probs.split(at_step.sum(dim=1).tolist())]


EXAMPLE_QUERY = "Sue lives in a fun neighborhood.  One weekend, the neighbors decided to play a prank on Sue.  On Friday morning, the neighbors placed 18 pink plastic flamingos out on Sue's front yard.  On Saturday morning, the neighbors took back one third of the flamingos, painted them white, and put these newly painted white flamingos back out on Sue's front yard.  Then, on Sunday morning, they added another 18 pink plastic flamingos to the collection. At noon on Sunday, how many more pink plastic flamingos were out than white plastic flamingos?"
EXAMPLE_STEPS = [
    "To find out how many more pink plastic flamingos were out than white plastic flamingos at noon on Sunday, we can break down the problem into steps. First, on Friday, the neighbors start with 18 pink plastic flamingos.",
    "On Saturday, they take back one third of the flamingos. Since there were 18 flamingos, (1/3 \\times 18 = 6) flamingos are taken back. So, they have (18 - 6 = 12) flamingos left in their possession. Then, they paint these 6 flamingos white and put them back out on Sue's front yard. Now, Sue has the original 12 pink flamingos plus the 6 new white ones. Thus, by the end of Saturday, Sue has (12 + 6 = 18) pink flamingos and 6 white flamingos.",
    "On Sunday, the neighbors add another 18 pink plastic flamingos to Sue's front yard. By the end of Sunday morning, Sue has (18 + 18 = 36) pink flamingos and still 6 white flamingos.",
    "To find the difference, subtract the number of white flamingos from the number of pink flamingos: (36 - 6 = 30). Therefore, at noon on Sunday, there were 30 more pink plastic flamingos out than white plastic flamingos. The answer is (\\boxed{30}).",
]
PUBLISHED = [1.0, 0.1904, 0.9766, 1.0]

got = prm_batch([sequence(EXAMPLE_QUERY, EXAMPLE_STEPS)])[0]
print("model card:", PUBLISHED)
print("this load: ", [round(x, 4) for x in got])
assert len(got) == len(PUBLISHED) and all(abs(a - b) < 0.1 for a, b in zip(got, PUBLISHED)), \
    "The PRM did not reproduce the model card's scores, so every score below would be wrong."

### Step 5: Score Every Chain

Every step of every chain gets a score, and each temperature is saved as `scored_T{temperature}_L2048.json`: the sweep's records, with each chain gaining `steps` and `prm`, one score per step. A chain that is an exact copy of another for the same problem (all of T 0.0) is run once and the score copied.

Progress is saved as it goes, so a run that disconnects picks up where it stopped when the cells are run again. For a fresh run, delete the `scored_*` files.

In [ ]:
def save_json(path, obj):
    with open(path + ".tmp", "w") as f:
        json.dump(obj, f)
    os.replace(path + ".tmp", path)


def length_batches(lengths):
    # Indices grouped longest first, each group under TOKENS_PER_BATCH padded tokens.
    order = sorted(range(len(lengths)), key=lambda i: -lengths[i])
    batch = []
    for i in order:
        if batch and lengths[batch[0]] * (len(batch) + 1) > TOKENS_PER_BATCH:
            yield batch
            batch = []
        batch.append(i)
    if batch:
        yield batch


def score_temperature(t):
    final = f"scored_T{t:.1f}_L{CAP}.json"
    if os.path.exists(final):
        with open(final) as f:
            traces[t] = json.load(f)
        print(f"T {t:.1f}: already scored")
        return

    partial = final + ".partial"
    done = {}
    if os.path.exists(partial):
        with open(partial) as f:
            done = json.load(f)

    # One job per distinct chain of a problem, named by where it first appears.
    first_seen, jobs = {}, []
    for rec in traces[t]:
        for i, chain in enumerate(rec["chains"]):
            chain["steps"] = split_steps(chain["text"])
            where = (rec["unique_id"], chain["text"])
            if where not in first_seen:
                first_seen[where] = f"{rec['unique_id']}#{i}"
                jobs.append((first_seen[where], rec["unique_id"], chain["steps"]))

    todo = [j for j in jobs if j[0] not in done]
    print(f"T {t:.1f}: {len(jobs):,} distinct chains, {len(todo):,} to score")
    sequences = [sequence(question[uid], steps) for _, uid, steps in todo]

    started = saved_at = time.time()
    for batch in tqdm(list(length_batches([len(s) for s in sequences])), desc=f"T {t:.1f}"):
        for i, scores in zip(batch, prm_batch([sequences[i] for i in batch])):
            key, _, steps = todo[i]
            assert len(scores) == len(steps), f"{key}: {len(scores)} scores for {len(steps)} steps"
            done[key] = scores
        if time.time() - saved_at > 60:
            save_json(partial, done)
            saved_at = time.time()

    for rec in traces[t]:
        for chain in rec["chains"]:
            chain["prm"] = done[first_seen[(rec["unique_id"], chain["text"])]]
    save_json(final, traces[t])
    if os.path.exists(partial):
        os.remove(partial)
    print(f"T {t:.1f}: saved {final} ({time.time() - started:.0f}s)")


for t in TEMPERATURES:
    score_temperature(t)

steps_scored = sum(len(c["prm"]) for t in TEMPERATURES for r in traces[t] for c in r["chains"])
print(f"\n{total:,} chains and {steps_scored:,} steps scored")

### Step 6: Best-of-N Accuracy

From a set of N chains, keep the one with the highest score; its answer is the final answer. A problem counts when that answer is right, averaged over the sets.

A chain's score is its **weakest step**. The last step's score and the product of all steps are computed beside it, from the same scores.

- A chain with no boxed answer cannot be kept. A set with no answer in it is wrong.
- A tie goes to the first chain in the set.

Saved to `prm_accuracy.csv`.

In [ ]:
RULES = {
    "weakest step": min,
    "last step": lambda s: s[-1],
    "product of steps": lambda s: sum(math.log(max(x, 1e-12)) for x in s),   # as a sum of logs
}

rng = np.random.default_rng(SEED)
right_at = defaultdict(list)   # (rule, temperature, N) -> one bool per set

for t in tqdm(TEMPERATURES, desc="Best-of-N"):
    by_id = {r["unique_id"]: r["chains"] for r in traces[t]}
    for p in problems:
        chains = by_id[p["unique_id"]]
        labels, right = tally(chains, p["answer"])
        answered = np.array([l is not None for l in labels])
        correct = np.array([l is not None and right[l] for l in labels])
        scores = {rule: np.array([f(c["prm"]) for c in chains]) for rule, f in RULES.items()}

        for n in BEST_OF_SIZES:
            for idx in chain_sets(n, rng):
                idx = np.array(idx)
                candidates = idx[answered[idx]]
                for rule, s in scores.items():
                    kept = candidates.size > 0 and correct[candidates[s[candidates].argmax()]]
                    right_at[rule, t, n].append(bool(kept))

accuracy = pd.DataFrame(
    [{"rule": rule, "temperature": t, "n_chains": n, "accuracy": float(np.mean(v))}
     for (rule, t, n), v in right_at.items()]
)
accuracy.to_csv("prm_accuracy.csv", index=False)

table = accuracy[accuracy.rule == "weakest step"].pivot(index="temperature", columns="n_chains", values="accuracy")
display(table.round(3))

### Step 7: Accuracy Against the Number of Chains

Best-of-N by weakest step, over 1, 2, 4, 8 and 16 chains, one line per temperature. The 0.0 line is greedy decoding: its 16 chains are identical, so it stays flat.

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 2, "font.size": 11,
})

by_n = accuracy[accuracy.rule == "weakest step"].pivot(index="n_chains", columns="temperature", values="accuracy")
best_t = float(by_n.loc[N_CHAINS].idxmax())   # a tie goes to the lower temperature
hot = [t for t in TEMPERATURES if t > 0]
shades = plt.cm.Blues(np.linspace(0.35, 1, len(hot)))

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(by_n.index, by_n[0.0], color="#0b0b0b", linestyle="--", marker="o", markersize=6, label="0.0 (greedy)")
for t, color in zip(hot, shades):
    ax.plot(by_n.index, by_n[t], color=color, marker="o", markersize=6,
            linewidth=3 if t == best_t else 1.5, label=f"{t:.1f}" + (" (best)" if t == best_t else ""))
ax.set_xscale("log", base=2)
ax.set_xticks(BEST_OF_SIZES, [str(n) for n in BEST_OF_SIZES])
ax.minorticks_off()
ax.set_xlabel("chains to choose from (N)")
ax.set_ylabel("accuracy")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Best-of-N by the PRM's weakest step, one line per temperature", loc="left")
ax.legend(title="temperature", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.savefig("fig_prm_accuracy_by_n.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"best temperature at N = {N_CHAINS}: {best_t} ({by_n.loc[N_CHAINS, best_t]:.1%})")

### Step 8: Best-of-16 Across Temperatures

Best-of-16 accuracy at each temperature, once for each way of turning step scores into one score.

In [ ]:
STYLE = {"weakest step": ("#2a78d6", "o"), "last step": ("#eb6834", "s"), "product of steps": ("#1baf7a", "^")}

top = accuracy[accuracy.n_chains == N_CHAINS].pivot(index="temperature", columns="rule", values="accuracy")[list(RULES)]

fig, ax = plt.subplots(figsize=(9, 5.5))
for rule in RULES:
    color, marker = STYLE[rule]
    ax.plot(top.index, top[rule], color=color, marker=marker, markersize=7, label=rule)

# Labels at the right end, nudged apart when the lines finish close together.
gap = 0.045 * (top.values.max() - top.values.min() + 0.02)
placed = []
for rule in top.iloc[-1].sort_values().index:
    y = max(top[rule].iloc[-1], placed[-1] + gap) if placed else top[rule].iloc[-1]
    placed.append(y)
    ax.annotate(rule, (top.index[-1], y), xytext=(10, 0), textcoords="offset points", va="center", color="#52514e")

ax.set_xticks(top.index, [f"{t:.1f}" for t in top.index])
ax.set_xlabel("temperature")
ax.set_ylabel(f"Best-of-{N_CHAINS} accuracy")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f"Best-of-{N_CHAINS} by the PRM, three ways to score a chain", loc="left")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3)
ax.margins(x=0.02)
plt.savefig("fig_prm_best_of_16.png", dpi=150, bbox_inches="tight")
plt.show()

display(top.round(3))

### Step 9: What Follows the First Low Step

A chain's break is its first step scored below a cut. The share after it is the chain's text that follows that step, by characters. Right answers are drawn beside wrong ones because a low step on a right chain is a false alarm.

The cut is a choice: 0.75 is the page's colour cut, with 0.5 and 0.9 beside it in the table. The figure and table use the best temperature from step 7. "Tokens after it" applies each chain's share to its token count.

Saved to `prm_breaks.csv` (every chain on the shared problems, every temperature, one row per cut) and `prm_break_summary.csv`.

In [ ]:
CUTS = [0.5, 0.75, 0.9]
MAIN_CUT = 0.75

rows = []
for t in TEMPERATURES:
    by_id = {r["unique_id"]: r["chains"] for r in traces[t]}
    for p in problems:
        chains = by_id[p["unique_id"]]
        labels, right = tally(chains, p["answer"])
        for i, (c, l) in enumerate(zip(chains, labels)):
            sizes = [len(s) for s in c["steps"]]
            for cut in CUTS:
                at = next((k for k, x in enumerate(c["prm"]) if x < cut), None)
                rows.append({
                    "temperature": t, "unique_id": p["unique_id"], "chain": i, "cut": cut,
                    "right_answer": l is not None and right[l], "tokens": c["tokens"], "n_steps": len(sizes),
                    "first_low_step": at,
                    "share_after": None if at is None else sum(sizes[at + 1 :]) / sum(sizes),
                })
breaks = pd.DataFrame(rows)
breaks.to_csv("prm_breaks.csv", index=False)

here = breaks[breaks.temperature == best_t]
GROUPS = {True: ("right answer", "#1baf7a"), False: ("wrong or no answer", "#2a78d6")}


def summarise(d):
    low = d[d.first_low_step.notna()]
    return {
        "chains": len(d),
        "with a low step": len(low) / len(d),
        "median share after it": low.share_after.median(),
        "tokens after it": (d.share_after.fillna(0) * d.tokens).sum() / d.tokens.sum(),
    }


summary = pd.DataFrame(
    [{"cut": cut, "chains are": GROUPS[is_right][0], **summarise(g)}
     for (cut, is_right), g in here.groupby(["cut", "right_answer"])]
)
summary.to_csv("prm_break_summary.csv", index=False)
display(summary.round(3))

main = here[here.cut == MAIN_CUT]
bins = np.linspace(0, 1, 11)
ticks = ["no low\nstep"] + [f"{a:.0f}–{b:.0f}" for a, b in zip(bins[:-1] * 100, bins[1:] * 100)]

fig, ax = plt.subplots(figsize=(10, 5.5))
for offset, (is_right, (label, color)) in zip((-0.2, 0.2), GROUPS.items()):
    g = main[main.right_answer == is_right]
    counts = np.histogram(g.share_after.dropna(), bins=bins)[0]
    heights = np.concatenate([[g.first_low_step.isna().sum()], counts]) / len(g)
    ax.bar(np.arange(len(ticks)) + offset, heights, width=0.38, color=color, label=f"{label} ({len(g):,} chains)")
ax.set_xticks(np.arange(len(ticks)), ticks)
ax.set_xlabel("share of the chain written after its first low step (%)")
ax.set_ylabel("share of chains")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis="x", visible=False)
ax.set_title(f"How much of a chain follows its first low step (below {MAIN_CUT}, temperature {best_t})", loc="left")
ax.legend(frameon=False, loc="upper center")
plt.savefig("fig_prm_first_low_step.png", dpi=150, bbox_inches="tight")
plt.show()